## **1. Problem Definition**

## **2. Data Collection**

In [ ]:
import os          # file/folder management
import random      # for shuffling dataset
import shutil      # copying/moving files
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import random

# Original paths
fake_dir  = "/content/drive/MyDrive/HAKIKI - FACES/fake faces/faces"
real_dir  = "/content/drive/MyDrive/HAKIKI - FACES/real faces"
others_dir= "/content/drive/MyDrive/HAKIKI - FACES/others"

# New root folder
dataset_root = "/content/drive/MyDrive/HAKIKI_dataset"

# Train/Val/Test splits
splits = ["train", "val", "test"]
classes = ["FAKE", "REAL", "OTHERS"]
for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(dataset_root, split, cls), exist_ok=True)

# Function to split and copy files
def split_and_copy(src_dir, cls_name, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    if not os.path.exists(src_dir):
        print(f"Source directory does not exist: {src_dir}")
        return

    files = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    n = len(files)

    if n == 0:
        print(f"No valid images found in {src_dir}")
        return

    random.shuffle(files)

    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    # Copy files
    for f in files[:train_end]:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dataset_root, "train", cls_name, f))
    for f in files[train_end:val_end]:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dataset_root, "val", cls_name, f))
    for f in files[val_end:]:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dataset_root, "test", cls_name, f))

    print(f"{cls_name}: {n} images split as -> train: {train_end}, val: {val_end-train_end}, test: {n-val_end}")

# Split each class
split_and_copy(fake_dir, "FAKE")
split_and_copy(real_dir, "REAL")
split_and_copy(others_dir, "OTHERS")

print("Dataset restructured into train/val/test successfully!")

FAKE: 376 images split as -> train: 263, val: 56, test: 57
REAL: 1250 images split as -> train: 875, val: 187, test: 188
OTHERS: 1127 images split as -> train: 788, val: 169, test: 170
Dataset restructured into train/val/test successfully!


## **3. Data Preparation**

**STEPS**
> Image Preprocessing: Clean and standardize the image data to enhance important features and improve the efficiency and accuracy of the model.

> Resizing: Adjust all images to a uniform size (e.g., 224x224 pixels) to ensure consistency, as models typically require fixed-size inputs.

> Normalization: Scale pixel values to a specific range, typically between 0 and 1, to make the data more suitable for neural network processing.

> Noise Reduction: Apply filters, if necessary, to eliminate unwanted variations in pixel intensity for cleaner images.

> Color Conversion (Optional): Convert images to grayscale to reduce computational complexity if color information is not a vital feature for classification.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Transforms
# -------------------------
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# -------------------------
# Load datasets
# -------------------------
train_dataset = datasets.ImageFolder("/content/drive/MyDrive/HAKIKI_dataset/train", transform=train_transform)
val_dataset   = datasets.ImageFolder("/content/drive/MyDrive/HAKIKI_dataset/val", transform=val_transform)
test_dataset  = datasets.ImageFolder("/content/drive/MyDrive/HAKIKI_dataset/test", transform=val_transform)

# -------------------------
# Dataloaders
# -------------------------
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

Train samples: 2568
Validation samples: 984
Test samples: 978
Classes: ['FAKE', 'OTHERS', 'REAL']


## **4. Feature Engineering**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Load EfficientNet-B0
# -----------------------------
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# -----------------------------
# Freeze feature extractor (for transfer learning)
# -----------------------------
for param in model.features.parameters():
    param.requires_grad = False

# -----------------------------
# Replace classifier for 3 classes
# -----------------------------
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(0.3),          # helps prevent overfitting
    nn.Linear(num_ftrs, 3)    # 3 classes: FAKE, REAL, OTHERS
)

# Move model to device
model = model.to(device)

# -----------------------------
# Loss function
# -----------------------------
criterion = nn.CrossEntropyLoss()

# -----------------------------
# Optimizer (only trains classifier first)
# -----------------------------
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)

Using device: cpu
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 112MB/s]


## **5. Model Training**

In [ ]:
from tqdm import tqdm
import torch
import os

num_epochs = 5  # Can increase later
best_val_acc = 0.0

# Path to save best weights
weights_path = "/content/drive/MyDrive/MODEL/efficientnet3class_best.pth"
os.makedirs(os.path.dirname(weights_path), exist_ok=True)

for epoch in range(num_epochs):

    # -----------------
    # TRAINING
    # -----------------
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    # -----------------
    # VALIDATION
    # -----------------
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [VAL]"):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"\nEpoch {epoch+1}:")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss/len(val_loader):.4f} | Val Acc:   {val_acc:.2f}%")

    # -----------------
    # SAVE BEST MODEL
    # -----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), weights_path)
        print(f"Saved best model weights with Val Acc: {best_val_acc:.2f}%\n")

Epoch 1/5 [VAL]: 100%|██████████| 31/31 [01:55<00:00,  3.71s/it]



Epoch 1:
Train Loss: 0.9048 | Train Acc: 62.15%
Val Loss:   0.7543 | Val Acc:   83.94%
Saved best model weights with Val Acc: 83.94%



Epoch 2/5 [VAL]: 100%|██████████| 31/31 [01:53<00:00,  3.66s/it]



Epoch 2:
Train Loss: 0.6779 | Train Acc: 82.79%
Val Loss:   0.6019 | Val Acc:   91.97%
Saved best model weights with Val Acc: 91.97%



Epoch 3/5 [VAL]: 100%|██████████| 31/31 [02:05<00:00,  4.04s/it]



Epoch 3:
Train Loss: 0.5465 | Train Acc: 87.15%
Val Loss:   0.5018 | Val Acc:   94.31%
Saved best model weights with Val Acc: 94.31%



Epoch 4/5 [VAL]: 100%|██████████| 31/31 [02:01<00:00,  3.93s/it]



Epoch 4:
Train Loss: 0.4631 | Train Acc: 89.37%
Val Loss:   0.4324 | Val Acc:   95.22%
Saved best model weights with Val Acc: 95.22%



Epoch 5/5 [VAL]: 100%|██████████| 31/31 [02:13<00:00,  4.32s/it]


Epoch 5:
Train Loss: 0.3951 | Train Acc: 91.55%
Val Loss:   0.3719 | Val Acc:   96.65%
Saved best model weights with Val Acc: 96.65%



## **6. Model Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        # Convert logits → probabilities
        probs = torch.softmax(outputs, dim=1)

        _, preds = torch.max(probs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# METRICS

print("\n=== Accuracy ===")
print(f"{accuracy_score(all_labels, all_preds)*100:.2f}%")

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(all_labels, all_preds))


=== Accuracy ===
96.65%

=== Classification Report ===
              precision    recall  f1-score   support

        FAKE       0.88      0.64      0.74        56
      OTHERS       0.99      0.98      0.99       440
        REAL       0.95      0.99      0.97       488

    accuracy                           0.97       984
   macro avg       0.94      0.87      0.90       984
weighted avg       0.97      0.97      0.96       984


=== Confusion Matrix ===
[[ 36   2  18]
 [  3 431   6]
 [  2   2 484]]


## **7. Model Deployment**

In [ ]:
import torch
from torchvision import models
import torch.nn as nn
import os

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Create MODEL folder in Google Drive if it doesn't exist
# -----------------------------
model_dir = "/content/drive/MyDrive/NEW MODEL"
os.makedirs(model_dir, exist_ok=True)

# -----------------------------
# Load EfficientNet-B0
# -----------------------------
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# -----------------------------
# Modify classifier for 3 classes: FAKE, REAL, OTHERS
# Match the architecture used during training (cell v5_WO1FCPSR8)
# -----------------------------
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(0.3),  # Re-add the Dropout layer as it was during training
    nn.Linear(num_ftrs, 3) # The final Linear layer
)

# -----------------------------
# Load trained weights (make sure weights exist)
# -----------------------------
weights_path = "/content/drive/MyDrive/MODEL/efficientnet3class_best.pth" # Corrected path
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)
model.eval()  # Set to evaluation mode

# -----------------------------
# Save only the weights
# -----------------------------
weights_save_path = os.path.join(model_dir, "efficientnet3class_weights.pth")
torch.save(model.state_dict(), weights_save_path)

# -----------------------------
# Save full deployment-ready model
# -----------------------------
full_model_save_path = os.path.join(model_dir, "efficientnet3class_full_model.pth")
torch.save(model, full_model_save_path)

print(f"Folder 'MODEL' created at {model_dir}")
print(f"Weights saved at: {weights_save_path}")
print(f"Full model saved at: {full_model_save_path}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 90.6MB/s]


Folder 'MODEL' created at /content/drive/MyDrive/NEW MODEL
Weights saved at: /content/drive/MyDrive/NEW MODEL/efficientnet3class_weights.pth
Full model saved at: /content/drive/MyDrive/NEW MODEL/efficientnet3class_full_model.pth


## **8. Test Model Performance**

In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to the full saved model
full_model_save_path = "/content/drive/MyDrive/MODEL/efficientnet3class_full_model.pth"

# Load the full model
# Ensure the model architecture matches the saved model
# (The full model save method handles this by saving the entire object)
loaded_model = torch.load(full_model_save_path, map_location=device, weights_only=False) # Added weights_only=False
loaded_model.eval() # Set the model to evaluation mode
loaded_model.to(device)

print("Model loaded successfully for testing.")

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = loaded_model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# METRICS
print("\n=== Test Accuracy ===")
print(f"{accuracy_score(all_labels, all_preds)*100:.2f}%")

print("\n=== Test Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

print("\n=== Test Confusion Matrix ===")
print(confusion_matrix(all_labels, all_preds))

Model loaded successfully for testing.

=== Test Accuracy ===
96.83%

=== Test Classification Report ===
              precision    recall  f1-score   support

        FAKE       0.90      0.65      0.76        57
      OTHERS       0.98      0.99      0.98       439
        REAL       0.97      0.99      0.98       482

    accuracy                           0.97       978
   macro avg       0.95      0.88      0.90       978
weighted avg       0.97      0.97      0.97       978


=== Test Confusion Matrix ===
[[ 37   7  13]
 [  3 433   3]
 [  1   4 477]]
